In [16]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()

model = init_chat_model(
    "google_genai:gemini-2.5-flash",
)

In [17]:
from typing import TypedDict, Annotated
from langchain.messages import AnyMessage
import operator

class MessagesState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

In [18]:
def llm_node(state: MessagesState):
    response = model.invoke(
        state["messages"]
    )

    return {"messages": [response]}

In [19]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

graph_builder = StateGraph(MessagesState)

graph_builder.add_node(llm_node)

graph_builder.add_edge(START, "llm_node")
graph_builder.add_edge("llm_node", END)

checkpointer = InMemorySaver()

graph = graph_builder.compile(checkpointer=checkpointer)

config = {"configurable": {"thread_id": "1"}}

In [20]:
from langchain.messages import HumanMessage

human_message = HumanMessage(content="내 이름은 김일남이야.")
result = graph.invoke({"messages": [human_message]}, config=config)

for m in result["messages"]:
    m.pretty_print()

================================ Human Message =================================

내 이름은 김일남이야.
================================== Ai Message ==================================

안녕하세요, 김일남 님! 만나서 반갑습니다.
무엇을 도와드릴까요?


In [21]:
human_message = HumanMessage(content="내 이름이 뭐야?")
result = graph.invoke({"messages": [human_message]}, config=config)

for m in result["messages"]:
    m.pretty_print()

================================ Human Message =================================

내 이름은 김일남이야.
================================== Ai Message ==================================

안녕하세요, 김일남 님! 만나서 반갑습니다.
무엇을 도와드릴까요?
================================ Human Message =================================

내 이름이 뭐야?
================================== Ai Message ==================================

김일남 님이세요.
